In [0]:
from pyspark.sql import DataFrame
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import(
    trim,
    col,
    row_number
)
from pyspark.sql.types import StringType

##### Importing from custom utility and test_utils package

In [0]:
from validation_utils.test_utils import Validations
from utils.custom_utils import Transformations
tr_obj = Transformations()
va_obj = Validations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.prd_info")

### Silver Layer Transformations

#### 1. Splitting product key
- The first five terms of the prd_key can be used as a foreign key for performing join with px_cat_g1v2 bronze table. So, split the prd_key into two columns and replace the '-' character with 'underscore' because px_cat_g1v2 doesn't have '-' but instead has 'underscore'.

In [0]:
df = (
    df.withColumn("cat_id",
    F.regexp_replace(
        F.substring(col("prd_key"), 1, 5), "-", "_")
    )
)
df = (
    df.withColumn("prd_key",
    F.substring(col("prd_key"), 7, F.length(col("prd_key"))))
)

#### 2. Renaming columns

In [0]:
rename_config = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
}

for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name)

#### 3. Removing records with no product_id
- Since there are no null product_id records so, transformation is not required

In [0]:
va_obj.null_check(df = df, primary_col = 'product_id')

#### 4. Removing product_id duplicates
- Since there are no duplicates so, transformation is not required

In [0]:
va_obj.duplicate_check(df = df, dedup_cols = ['product_id'], cdc = 'start_date')

#### 5. Trimming

In [0]:
df = tr_obj.trim_columns(df)

#### 6. Replacing nulls in product_cost with 0

In [0]:
df = df.withColumn("product_cost", F.coalesce(col("product_cost"), F.lit(0)))

#### 7. Normalization

In [0]:
df = (
    df.withColumn(
        "product_line",
        F.when(F.upper(col("product_line")) == "R", "Road")
        .when(F.upper(col("product_line")) == "S", "Other Sales")
        .when(F.upper(col("product_line")) == "M", "Mountain")
        .when(F.upper(col("product_line")) == "T", "Touring")
        .otherwise("n/a")
    )
)

#### 8. Cleaning the start_date and end_date
-- Conditions:
- Start_date must be less than end_date
- For same product [name and category], it cannot have different price for same time period
- Start date cannot be null 

In [0]:
window_spec = Window.partitionBy(col("product_number")).orderBy(col("start_date").asc())
df = df.withColumn("end_date", (F.lead(col("start_date")).over(window_spec)) - 1)

#### 9. Adding ingestion_ts column

In [0]:
df = tr_obj.add_ingestion_timestamp(df)

#### DataFrame sanity check

In [0]:
df.limit(10).display()

#### 10. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.crm_products"

if spark.catalog.tableExists(target_table):
    tr_obj.upsert(
        spark = spark,
        df = df,
        key_cols = ["product_id"],
        table = "crm_products",
        cdc = "start_date"
    )
else:
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
select *
from lakehouse.silver.crm_products
limit 10